# Phase 2 — Condition 4 (Pretraining): Self-Supervised Swin on Clean Brain Data

Restores the original research title — **Self-Supervised 3D Swin Transformer** — with the SSL actually working this time. Trains the Swin encoder with self-supervision on the clean `processed_mri_scans_brain` volumes, using industry-standard collapse/overfitting defenses.

**Two objectives, ablated:**
- **VICReg** (Bardes/Ponce/LeCun ICLR'22) — variance-invariance-covariance regularization; prevents collapse without negative pairs, ideal for small batch / small N.
- **Fixed-MAE** — the original masked-autoencoder idea, but with patch masking (not voxel) and encoder-only reconstruction (no skip leakage).

**Anti-overfitting / anti-collapse stack (all citable):**
VICReg variance+covariance terms · orthogonality regularization on encoder weights · heavy 3D augmentation · weight decay · early stopping · live collapse monitor (embedding std per epoch).

**Output:** two encoder checkpoints saved to Drive (`ssl_vicreg_encoder.pth`, `ssl_mae_encoder.pth`) + pretraining logs. Notebook 4-eval then plugs each into the frozen tri-modal setup (the 3b recipe) and compares against 3b's CT-pretrained gate of 0.141.

> Colab Pro: set Runtime → GPU (A100/L4). Pretraining each objective is ~1-2 hrs. The collapse monitor will ABORT early and loudly if embeddings collapse, so you won't waste compute on a dead run.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip install -q "monai==1.5.0" nibabel torchio imbalanced-learn
import os
os._exit(0)  # restart for clean binaries


## 1. Verify env (after restart)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import torch, numpy, monai
print('NumPy', numpy.__version__, '| MONAI', monai.__version__, '| torch', torch.__version__)
from monai.networks.nets import SwinUNETR
assert torch.cuda.is_available(), 'Enable GPU (Colab Pro: A100/L4 recommended).'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Config & paths

In [ ]:
import os, json, numpy as np, torch, datetime
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchio as tio
from tqdm.notebook import tqdm
device=torch.device('cuda')

DRIVE='/content/drive/My Drive/ADNI_NewDS/'
RESULTS=os.path.join(DRIVE,'results')
BRAIN_DIR=os.path.join(RESULTS,'processed_mri_scans_brain')   # clean volumes from Cond 3a
SPLITS=os.path.join(RESULTS,'patient_id_splits.npz')
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study')
SSL_DIR=os.path.join(PHASE2,'ssl_pretrain'); os.makedirs(SSL_DIR, exist_ok=True)
SSL_LOG=os.path.join(SSL_DIR,'ssl_pretrain_log.jsonl')

IMG_SIZE=(96,96,96); FEATURE_SIZE=48; VIT_DIM=FEATURE_SIZE*16  # 768
SEED=42; torch.manual_seed(SEED); np.random.seed(SEED)

# SSL pretraining uses ALL non-test volumes (train+val) — NO labels needed.
sp=np.load(SPLITS, allow_pickle=True)
ssl_pids=list(sp['pids_train'])+list(sp['pids_val'])   # hold out test set entirely
# verify files
ssl_pids=[p for p in ssl_pids if os.path.exists(os.path.join(BRAIN_DIR,f'{p}_processed.npy'))]
print(f'SSL pretraining volumes (train+val, unlabeled): {len(ssl_pids)}')
assert len(ssl_pids)>100, 'Too few volumes — check BRAIN_DIR path.'


## 3. Two-view augmentation pipeline
The augmentations ARE the self-supervision signal: the model must produce similar embeddings for two different augmented views of the same brain. Heavy augmentation also doubles as overfitting defense.

In [ ]:
# Strong but anatomy-preserving 3D augmentations
aug=tio.Compose([
    tio.RandomAffine(scales=(0.9,1.1), degrees=10, translation=5, p=0.8),
    tio.RandomFlip(axes=('LR',), p=0.5),
    tio.RandomNoise(std=(0,0.05), p=0.3),
    tio.RandomBiasField(coefficients=0.3, p=0.3),
    tio.RandomGamma(log_gamma=(-0.3,0.3), p=0.3),
    tio.Resize(IMG_SIZE),
    tio.ZNormalization(masking_method=tio.ZNormalization.mean),
])

class TwoViewDataset(Dataset):
    def __init__(self, pids): self.pids=pids
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        vol=np.load(os.path.join(BRAIN_DIR, f'{self.pids[i]}_processed.npy'))
        subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
        v1=aug(subj).mri.tensor; v2=aug(subj).mri.tensor   # two independent views
        return v1, v2

BATCH=8   # Colab Pro A100 can handle this for frozen-size encoder fwd; lower to 4 if OOM
ssl_loader=DataLoader(TwoViewDataset(ssl_pids), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
print('SSL batches/epoch:', len(ssl_loader))


## 4. Encoder + projector, VICReg loss, orthogonality regularizer
VICReg coefficients are the paper defaults (λ=25 invariance, μ=25 variance, ν=1 covariance, γ=1).

In [ ]:
from monai.networks.nets import SwinUNETR

class SwinEncoder(nn.Module):
    """Wraps MONAI SwinUNETR's swinViT; outputs a pooled VIT_DIM vector."""
    def __init__(self):
        super().__init__()
        self.swin=SwinUNETR(in_channels=1, out_channels=1, feature_size=FEATURE_SIZE, use_checkpoint=True).swinViT
        self.pool=nn.AdaptiveAvgPool3d(1)
    def forward(self,x):
        f=self.swin(x)[-1]
        return self.pool(f).flatten(1)   # [B, VIT_DIM]

class Projector(nn.Module):
    """VICReg projector: expands to a high-dim space where the loss is applied."""
    def __init__(self, in_dim=VIT_DIM, hid=2048, out=2048):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hid), nn.BatchNorm1d(hid), nn.ReLU(),
                               nn.Linear(hid,hid), nn.BatchNorm1d(hid), nn.ReLU(),
                               nn.Linear(hid,out))
    def forward(self,x): return self.net(x)

def off_diagonal(m):
    n,_=m.shape; return m.flatten()[:-1].view(n-1,n+1)[:,1:].flatten()

def vicreg_loss(z1, z2, lam=25.0, mu=25.0, nu=1.0, gamma=1.0, eps=1e-4):
    # invariance
    inv=F.mse_loss(z1,z2)
    # variance (hinge on std per dim)
    std1=torch.sqrt(z1.var(dim=0)+eps); std2=torch.sqrt(z2.var(dim=0)+eps)
    var=torch.mean(F.relu(gamma-std1))+torch.mean(F.relu(gamma-std2))
    # covariance (off-diagonal of cov matrix)
    N,D=z1.shape
    z1c=z1-z1.mean(0); z2c=z2-z2.mean(0)
    cov1=(z1c.T@z1c)/(N-1); cov2=(z2c.T@z2c)/(N-1)
    cov=off_diagonal(cov1).pow(2).sum()/D + off_diagonal(cov2).pow(2).sum()/D
    loss=lam*inv + mu*var + nu*cov
    return loss, {'inv':inv.item(),'var':var.item(),'cov':cov.item()}

def orthogonality_reg(model, beta=1e-4):
    """Soft orthogonality on linear/conv weight matrices (prevents dimensional collapse)."""
    reg=0.0
    for m in model.modules():
        if isinstance(m,(nn.Linear,)):
            W=m.weight; WT=W@W.t(); I=torch.eye(WT.size(0),device=W.device)
            reg=reg+((WT-I)**2).sum()
    return beta*reg
print('Loss + regularizers defined.')


## 5. VICReg pretraining (with live collapse monitor)
The monitor computes embedding std each epoch. If it falls toward 0 (collapse), training ABORTS early so no compute is wasted.

In [ ]:
def pretrain_vicreg(epochs=80, patience=15, ortho=True):
    enc=SwinEncoder().to(device); proj=Projector().to(device)
    opt=torch.optim.AdamW(list(enc.parameters())+list(proj.parameters()), lr=1e-4, weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best=1e9; wait=0; log=[]
    ENC_PATH=os.path.join(SSL_DIR,'ssl_vicreg_encoder.pth')
    for ep in range(epochs):
        enc.train(); proj.train(); tot=0; comps={'inv':0,'var':0,'cov':0}; embs=[]
        for v1,v2 in tqdm(ssl_loader, desc=f'VICReg {ep+1}/{epochs}', leave=False):
            v1,v2=v1.to(device),v2.to(device)
            z1=proj(enc(v1)); z2=proj(enc(v2))
            loss,c=vicreg_loss(z1,z2)
            if ortho: loss=loss+orthogonality_reg(proj)
            opt.zero_grad(); loss.backward(); opt.step()
            tot+=loss.item(); [comps.__setitem__(k,comps[k]+c[k]) for k in c]
            embs.append(enc(v1).detach())
        sched.step()
        # collapse monitor: std of encoder embeddings across the epoch
        E=torch.cat(embs); emb_std=E.std(0).mean().item()
        avg=tot/len(ssl_loader)
        log.append({'epoch':ep+1,'loss':avg,'emb_std':emb_std,**{k:comps[k]/len(ssl_loader) for k in comps}})
        print(f'Ep{ep+1}: loss={avg:.3f} emb_std={emb_std:.4f} inv={comps["inv"]/len(ssl_loader):.3f} var={comps["var"]/len(ssl_loader):.3f}')
        # ABORT if collapsing
        if emb_std < 0.01 and ep>5:
            print('  COLLAPSE DETECTED (emb_std<0.01). Aborting.'); break
        if avg<best: best=avg; torch.save(enc.state_dict(), ENC_PATH); wait=0; print('  saved best encoder')
        else:
            wait+=1
            if wait>=patience: print('  early stop'); break
    json.dump(log, open(os.path.join(SSL_DIR,'vicreg_log.json'),'w'), indent=2)
    return ENC_PATH, log

vicreg_path, vicreg_log = pretrain_vicreg()
print('VICReg encoder saved:', vicreg_path)


## 6. Fixed-MAE pretraining (patch masking, encoder-only — audit bugs fixed)
The original MAE collapsed from voxel-masking + skip leakage. This version masks whole patches and reconstructs from the encoder bottleneck only.

In [ ]:
class PatchMAE(nn.Module):
    def __init__(self, mask_ratio=0.6, patch=16):
        super().__init__()
        self.enc=SwinEncoder()
        self.mask_ratio=mask_ratio; self.patch=patch
        # lightweight decoder from pooled embedding back to a downsampled volume
        self.dec=nn.Sequential(nn.Linear(VIT_DIM,512), nn.ReLU(), nn.Linear(512, 6*6*6))
    def patch_mask(self, x):
        B,C,D,H,W=x.shape; p=self.patch
        md_,mh,mw=D//p,H//p,W//p
        mask=(torch.rand(B,md_,mh,mw,device=x.device)<self.mask_ratio).float()
        mask=mask.repeat_interleave(p,1).repeat_interleave(p,2).repeat_interleave(p,3)[:,None]
        return x*(1-mask)
    def forward(self,x):
        target=F.adaptive_avg_pool3d(x,(6,6,6)).flatten(1)
        masked=self.patch_mask(x)
        emb=self.enc(masked)
        recon=self.dec(emb)
        return F.mse_loss(recon,target), emb

def pretrain_mae(epochs=80, patience=15, ortho=True):
    model=PatchMAE().to(device)
    opt=torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best=1e9; wait=0; log=[]
    ENC_PATH=os.path.join(SSL_DIR,'ssl_mae_encoder.pth')
    for ep in range(epochs):
        model.train(); tot=0; embs=[]
        for v1,_ in tqdm(ssl_loader, desc=f'MAE {ep+1}/{epochs}', leave=False):
            v1=v1.to(device)
            loss,emb=model(v1)
            if ortho: loss=loss+orthogonality_reg(model.dec)
            opt.zero_grad(); loss.backward(); opt.step()
            tot+=loss.item(); embs.append(emb.detach())
        sched.step()
        E=torch.cat(embs); emb_std=E.std(0).mean().item(); avg=tot/len(ssl_loader)
        log.append({'epoch':ep+1,'loss':avg,'emb_std':emb_std})
        print(f'Ep{ep+1}: recon_loss={avg:.4f} emb_std={emb_std:.4f}')
        if emb_std<0.01 and ep>5: print('  COLLAPSE DETECTED. Aborting.'); break
        if avg<best: best=avg; torch.save(model.enc.state_dict(), ENC_PATH); wait=0; print('  saved best encoder')
        else:
            wait+=1
            if wait>=patience: print('  early stop'); break
    json.dump(log, open(os.path.join(SSL_DIR,'mae_log.json'),'w'), indent=2)
    return ENC_PATH, log

mae_path, mae_log = pretrain_mae()
print('MAE encoder saved:', mae_path)


## 7. Pretraining summary

In [ ]:
def final_std(log): return log[-1]['emb_std'] if log else float('nan')
print('='*56)
print('SSL PRETRAINING COMPLETE')
print(f'  VICReg: final emb_std={final_std(vicreg_log):.4f}  (>0.01 = no collapse)')
print(f'  MAE   : final emb_std={final_std(mae_log):.4f}')
print('  Encoders saved to', SSL_DIR)
entry={'timestamp':datetime.datetime.now(datetime.UTC).isoformat(),
       'vicreg_final_emb_std':final_std(vicreg_log),'mae_final_emb_std':final_std(mae_log),
       'ssl_pids':len(ssl_pids),'batch':BATCH}
with open(SSL_LOG,'a') as f: f.write(json.dumps(entry)+'\n')
print('\nNext: run Notebook 4-eval to plug each encoder into the tri-modal model and compare vs 3b (gate 0.141).')
print('Paste this summary back to Claude.')
